In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics
!pip install -q keras
!pip install deep_sort_realtime
!pip install cvzone
!pip install opencv-python-headless




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 31.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12


In [ ]:
import cv2
import cvzone
from collections import defaultdict
import math
from ultralytics import YOLO
from google.colab import files


def select_file(file_type):
    """Helper function to upload a file in Colab."""
    uploaded = files.upload()
    file_path = next(iter(uploaded))
    print(f"Selected {file_type}: {file_path}")
    return file_path


def select_output_file():
    """Manually define the output file in Colab."""
    output_path = "/content/output_video.mp4"
    print(f"Output will be saved at: {output_path}")
    return output_path


class PeopleCounter:
    def __init__(self, model_path='yolov8n.pt', conf_threshold=0.4):  # Corrected __init__ method
        self.model = YOLO(model_path)
        self.conf_threshold = conf_threshold

    def is_left_of_line(self, point, line_start, line_end):
        """Check if a point is on the left side of a slant line."""
        (x, y) = point
        (x1, y1), (x2, y2) = line_start, line_end
        return (x2 - x1) * (y - y1) - (y2 - y1) * (x - x1) > 0

    def process_video(self, video_path, output_path, line_start, line_end, distance_threshold=60):
        vid = cv2.VideoCapture(video_path)
        frame_width = int(vid.get(3))
        frame_height = int(vid.get(4))
        fps = int(vid.get(cv2.CAP_PROP_FPS))

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

        mask_path = '/content/drive/MyDrive/project/newmask.png'
        mask = cv2.imread(mask_path)

        tracker = defaultdict(dict)
        next_id = 0
        people_in = 0
        people_out = 0

        while True:
            success, img = vid.read()
            if not success:
                break

            imgRegion = cv2.bitwise_and(img, mask)
            results = self.model(imgRegion, stream=True)
            current_ids = []

            for r in results:
                boxes = r.boxes
                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0]
                    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
                    w, h = x2 - x1, y2 - y1
                    cls = int(box.cls[0])
                    conf = box.conf[0]

                    if cls == 0 and conf > self.conf_threshold:
                        cvzone.cornerRect(img, (x1, y1, w, h), l=9)
                        cvzone.putTextRect(img, f'{conf:.2f}', (max(0, x1), max(35, y1)), scale=0.8, thickness=1, offset=3)
                        cx, cy = x1 + w // 2, y1 + h // 2
                        matched_id = None

                        for person_id, person_info in tracker.items():
                            px, py = person_info['centroid']
                            distance = math.hypot(cx - px, cy - py)
                            if distance < distance_threshold:
                                matched_id = person_id
                                break

                        if matched_id is None:
                            matched_id = next_id
                            next_id += 1

                        current_ids.append(matched_id)
                        tracker[matched_id]['centroid'] = (cx, cy)
                        current_side = self.is_left_of_line((cx, cy), line_start, line_end)

                        if 'side' not in tracker[matched_id]:
                            tracker[matched_id]['side'] = current_side
                            tracker[matched_id]['counted_in'] = False
                            tracker[matched_id]['counted_out'] = False

                        previous_side = tracker[matched_id]['side']

                        if previous_side != current_side:
                            if previous_side and not tracker[matched_id]['counted_in']:
                                people_in += 1
                                tracker[matched_id]['counted_in'] = True
                                tracker[matched_id]['counted_out'] = False
                                print(f"Person {matched_id} entered. Total In: {people_in}")
                            elif not previous_side and not tracker[matched_id]['counted_out']:
                                people_out += 1
                                tracker[matched_id]['counted_out'] = True
                                tracker[matched_id]['counted_in'] = False
                                print(f"Person {matched_id} exited. Total Out: {people_out}")

                            tracker[matched_id]['side'] = current_side

            for person_id in list(tracker.keys()):
                if person_id not in current_ids:
                    del tracker[person_id]

            cv2.line(img, line_start, line_end, (0, 255, 0), 2)

            cvzone.putTextRect(img, f'OUT: {people_in}', (50, 50), scale=1, thickness=2, offset=10)
            cvzone.putTextRect(img, f'IN: {people_out}', (200, 50), scale=1, thickness=2, offset=10)

            out.write(img)

        vid.release()
        out.release()
        cv2.destroyAllWindows()

        print(f"Video saved at {output_path}")


if __name__ == "__main__":
    counter = PeopleCounter()

    video_path = select_file("Video")
    output_path = select_output_file()

    line_start = (712, 656)
    line_end = (1340, 275)

    counter.process_video(video_path, output_path, line_start, line_end)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
files.download('/content/output_video.mp4')



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>